LangChain

In [1]:
from langchain_community.document_loaders import WebBaseLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_community.vectorstores import FAISS
from langchain.memory import ConversationBufferMemory
from langchain.chains import ConversationalRetrievalChain

from langchain_google_genai import ChatGoogleGenerativeAI

from langchain_community.embeddings import HuggingFaceEmbeddings



In [2]:
from dotenv import load_dotenv
import os

In [3]:
load_dotenv()
os.environ["GOOGLE_API_KEY"] = os.getenv("GEMINI_API_KEY")

In [4]:
url = "https://en.wikipedia.org/wiki/Dhurandhar:_The_Revenge"
loader = WebBaseLoader(url)
raw_document = loader.load()
# had to do "pip install beautifulsoup4"

In [5]:
text_splitter = RecursiveCharacterTextSplitter()

In [6]:
docuemnts = text_splitter.split_documents(raw_document)

In [7]:
embeddings = HuggingFaceEmbeddings(model_name="all-MiniLM-L6-v2")

In [9]:
VECTORSTORE = FAISS.from_documents(docuemnts,embeddings)
# Loader → Splitter → HuggingFace Embeddings → FAISS → Gemini LLM

In [8]:
llm = ChatGoogleGenerativeAI(
    model="gemini-3-flash-preview",
    temperature=0.7
)

In [10]:
memory = ConversationBufferMemory(memory_key="chat_history", return_messages=True)


In [11]:
qa = ConversationalRetrievalChain.from_llm(
    llm=llm,
    retriever=VECTORSTORE.as_retriever(),
    memory=memory
)

In [12]:
query = "what is the Dhurandhar movie about?"

In [13]:
result = qa({"question":query})

h:\Conda\envs\LLM_env\Lib\site-packages\langchain_core\_api\deprecation.py:119: LangChainDeprecationWarning: The method `Chain.__call__` was deprecated in langchain 0.1.0 and will be removed in 0.3.0. Use invoke instead.
  warn_deprecated(


In [14]:
result['answer']

'Based on the provided context, the *Dhurandhar* film series (consisting of *Dhurandhar* released in 2025 and *Dhurandhar: The Revenge* released in 2026) is a political action-thriller franchise directed by Aditya Dhar. \n\nKey details about the movies include:\n\n*   **Plot and Themes:** The films are spy action thrillers that focus on themes of revenge, organized crime, and India–Pakistan relations. They involve the Research & Analysis Wing (R&AW), the Intelligence Bureau (IB), and the Indian Army. The plot of the sequel, *Dhurandhar: The Revenge*, specifically follows the protagonist Hamza (played by Ranveer Singh) on a violent mission to Pakistan.\n*   **Real-Life Inspirations:** The movies are based on or inspired by actual events, including the 2008 Mumbai attacks, the Balochistan insurgency, encounters in Pakistan, arms trafficking, and the issue of counterfeit money (linked to demonetization). It also features depictions of real-life figures such as Asif Ali Zardari and Nawaz S

In [15]:
query2 = "What is the date today?"
result = qa({"question":query2})
result['answer']

"I don't know."

In [16]:
query2 = "Who was first prime Minister of India?"
result = qa({"question":query2})
result['answer']

"I don't know. The provided text discusses the film *Dhurandhar* and its sequel, mentioning characters based on modern political figures like Narendra Modi, Nawaz Sharif, and Asif Ali Zardari, but it does not contain information about the first Prime Minister of India."